[STARTER] Udaplay Project
Part 01 - Offline RAG
In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder project/starter/games. Each file will become a document in the collection you'll create. Example.:

{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}

Setup

In [15]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [16]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"

In [17]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [19]:
# TODO: Load environment variables
load_dotenv("config.env")

True

VectorDB Instance

In [20]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chroma_db")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [22]:
# TODO: Create a collection
# Choose any name you want
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL"),
    model_name="text-embedding-3-small"
)
 
collection = chroma_client.get_or_create_collection(
    name="games",
    embedding_function=embedding_fn
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [23]:
print("Collection Name:", collection.name)
print("Docuements:", collection.count())

Collection Name: games
Docuements: 15


In [24]:
import os
import json
 
data_dir = "games"
 
for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue
 
    file_path = os.path.join(data_dir, file_name)
 
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)
 
    content = (
        f"Platform: {game.get('Platform', '')}\n"
        f"Name: {game.get('Name', '')}\n"
        f"Genre: {game.get('Genre', '')}\n"
        f"Publisher: {game.get('Publisher', '')}\n"
        f"YearOfRelease: {game.get('YearOfRelease', '')}\n"
        f"Summary: {game.get('Summary', '')}"
    )
    doc_id = os.path.splitext(file_name)[0]
 
    collection.upsert(
        ids=[doc_id],
        documents=[content],
        metadatas=[{
            "name": game.get("Name", ""),
            "platform": game.get("Platform", "")
        }]
    )
 
print("Documents added:", collection.count())

Documents added: 15


In [25]:
#temp code
import os
import json
 
for file_name in sorted(os.listdir("games")):
    if file_name.endswith(".json"):
        file_path = os.path.join("games", file_name)
 
        with open(file_path, "r", encoding="utf-8") as f:
            game = json.load(f)
 
        if "Pokémon Gold and Silver" in str(game):
            print("FILE:", file_name)
            print("FULL RECORD:")
            print(game)

FILE: 006.json
FULL RECORD:
{'Name': 'Pokémon Gold and Silver', 'Platform': 'Game Boy Color', 'Genre': 'Role-playing', 'Publisher': 'Nintendo', 'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.', 'YearOfRelease': 1999}


In [26]:
results = collection.query(
    query_texts=["When Pokémon Gold and Silver was released?"],
    n_results=5
)
 
print(results["documents"][0][0])

Platform: Game Boy Color
Name: Pokémon Gold and Silver
Genre: Role-playing
Publisher: Nintendo
YearOfRelease: 1999
Summary: 


In [27]:
# Test semantic search on the UdaPlay vector database
 
query = "Which football video games are available?"
 
results = collection.query(
    query_texts=[query],
    n_results=3
)
 
print("Semantic Search Query:")
print(query)
 
print("\nRetrieved Documents:")
 
for i, document in enumerate(results["documents"][0], start=1):
    print(f"\nResult {i}:")
    print(document)

Semantic Search Query:
Which football video games are available?

Retrieved Documents:

Result 1:
Platform: PlayStation 3
Name: Gran Turismo 5
Genre: Racing
Publisher: Sony Computer Entertainment
YearOfRelease: 2010
Summary: 

Result 2:
Platform: Xbox 360
Name: Kinect Adventures!
Genre: Party
Publisher: Microsoft Game Studios
YearOfRelease: 2010
Summary: 

Result 3:
Platform: Wii
Name: Wii Sports
Genre: Sports
Publisher: Nintendo
YearOfRelease: 2006
Summary: 


In [28]:
# Another semantic search example
 
query = "Find a game published by Electronic Arts"
 
results = collection.query(
    query_texts=[query],
    n_results=3
)
 
print("Semantic Search Query:")
print(query)
 
print("\nRetrieved Documents:")
 
for i, document in enumerate(results["documents"][0], start=1):
    print(f"\nResult {i}:")
    print(document)

Semantic Search Query:
Find a game published by Electronic Arts

Retrieved Documents:

Result 1:
Platform: PlayStation 1
Name: Gran Turismo
Genre: Racing
Publisher: Sony Computer Entertainment
YearOfRelease: 1997
Summary: 

Result 2:
Platform: Xbox 360
Name: Kinect Adventures!
Genre: Party
Publisher: Microsoft Game Studios
YearOfRelease: 2010
Summary: 

Result 3:
Platform: PlayStation 3
Name: Gran Turismo 5
Genre: Racing
Publisher: Sony Computer Entertainment
YearOfRelease: 2010
Summary: 
